# Run RAG Query

This notebook is to run the RAG query. 

In [34]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")


Imports loaded.


# Reload Vector DB 

In [35]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15765.76it/s]


In [36]:
from langchain_community.vectorstores import FAISS

# Paths for each vector DB
fixed_vector_path = "fixed_rag_faiss_index"
structure_path = "faiss_structure_chunks"
sentence_path = "faiss_sentence_chunks"
semantic_path = "faiss_semantic_chunks"

# Load each vector DB

fixed_vector_db = FAISS.load_local(
    fixed_vector_path,
    embeddings,
    allow_dangerous_deserialization=True
)

structure_vector_db = FAISS.load_local(
    structure_path,
    embeddings,
    allow_dangerous_deserialization=True
)

sentence_vector_db = FAISS.load_local(
    sentence_path,
    embeddings,
    allow_dangerous_deserialization=True
)

semantic_vector_db = FAISS.load_local(
    semantic_path,
    embeddings,
    allow_dangerous_deserialization=True
)

print("All vector DBs reloaded successfully.")

print("Vector DB reloaded from local folder.")

All vector DBs reloaded successfully.
Vector DB reloaded from local folder.


In [37]:
# Store all 4 vector DBs in one dictionary

vector_dbs = {
    "fixed_size": fixed_vector_db,
    "document_structure": structure_vector_db,
    "sentence": sentence_vector_db,
    "semantic": semantic_vector_db
}

print("Available vector DBs:", list(vector_dbs.keys()))

Available vector DBs: ['fixed_size', 'document_structure', 'sentence', 'semantic']


In [38]:
# Load local Hugging Face model for answer generation
# TinyLlama is a causal model, so we must format the prompt as chat and decode ONLY new tokens.

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Use CPU-safe loading. If you have a GPU, you can change this later.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=None
)

model.eval()
print("LLM loaded.")


Loading weights: 100%|██████████| 201/201 [00:04<00:00, 42.63it/s]

LLM loaded.


In [39]:
def llm(prompt: str, max_new_tokens: int = 350) -> str:
    """
    Generate an answer using TinyLlama.
    Important fix: decode only the newly generated tokens, not the full prompt.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful RAG assistant. Answer only from the provided context. "
                "Use citations exactly as provided, such as [1] or [1, p. 3]. "
                "If the context does not answer the question, say that the documents do not contain enough information."
                "Do not stop mid-sentence."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # CRITICAL FIX: remove the prompt tokens from the output
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


In [40]:
def retrieve_chunks(query: str, vector_db, k: int = 4):
    """Retrieve top-k chunks from a selected FAISS vector database."""
    return vector_db.similarity_search(query, k=k)


# Quick retrieval test
test_chunks = retrieve_chunks(
    "What is copyright?",
    vector_dbs["fixed_size"],
    k=4
)

print(f"Retrieved {len(test_chunks)} chunks")

for i, doc in enumerate(test_chunks, start=1):
    print(f"\nChunk {i}")
    print("Source:", doc.metadata.get("source_file", "Unknown file"))
    print("Page:", doc.metadata.get("page", "Unknown page"))
    print("Chunking:", doc.metadata.get("chunking_method", "fixed_size"))
    print("Citation:", doc.metadata.get("ieee_citation", "Unknown citation"))
    print("Preview:", doc.page_content[:250].replace("\n", " "))

Retrieved 4 chunks

Chunk 1
Source: 3614407.3643696.pdf
Page: 8
Chunking: fixed_size
Citation: Katherine Lee, A. Feder Cooper, James Grimmelmann, "Talkin' 'Bout AI Generation," Proceedings of the Symposium on Computer Science and Law, ACM, 2024, doi: 10.1145/3614407.3643696.
Preview: and to noncommercial use (e.g., illustrating an academic article on generative AI). Some outputs will be put to favored purposes like education and news reporting, while other outputs will be put to run-of-the-mill entertainment purposes.75 Factor Tw

Chunk 2
Source: 3614407.3643696.pdf
Page: 5
Chunking: fixed_size
Citation: Katherine Lee, A. Feder Cooper, James Grimmelmann, "Talkin' 'Bout AI Generation," Proceedings of the Symposium on Computer Science and Law, ACM, 2024, doi: 10.1145/3614407.3643696.
Preview: right law to the generative-AI supply chain. We address issues of rights (Section 3.2), infringement (Sections 3.3 & 3.4), and fair use (Section 3.5). We defer discussion of safe harbors, licenses, 

In [41]:
def rag_query(question: str, vector_db, method_name: str, k: int = 5) -> dict:
    """
    RAG pipeline for one chunking method.
    """
    try:
        retrieved_docs = retrieve_chunks(question, vector_db, k=k)

        if not retrieved_docs:
            return {
                "method": method_name,
                "question": question,
                "answer": "I could not retrieve any relevant document chunks.",
                "retrieved_docs": []
            }

        context_blocks = []

        for i, doc in enumerate(retrieved_docs, start=1):
            page = doc.metadata.get("page", "Unknown page")
            source_file = doc.metadata.get("source_file", "Unknown file")
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            context_blocks.append(
                f"""
[{i}]
File: {source_file}
Page: {page}
Citation: {citation}

Text:
{doc.page_content[:700]}
"""
            )

        context = "\n\n---\n\n".join(context_blocks)

        prompt = f"""
Answer the question using only the sources below.
Use citations like [1] or [2].
Keep the answer concise.
Do not stop mid-sentence.

Question:
{question}

Sources:
{context}

Answer:
"""

        answer = llm(prompt)

        return {
            "method": method_name,
            "question": question,
            "answer": answer,
            "retrieved_docs": retrieved_docs
        }

    except Exception as e:
        return {
            "method": method_name,
            "question": question,
            "error": str(e)
        }

In [42]:
# Test one method

def print_rag_query_answer(question, vdb, method_name, k=2): 
    result = rag_query(
        question,
        vdb,
        method_name,
        k=2
    )

    if "error" in result:
        print("RAG failed:", result["error"])
    else:
        print("Method:", result["method"])
        print("Question:", result["question"])
        print("\nAnswer:\n")
        print(result["answer"])

        print("\nREFERENCES:\n")

        seen = set()

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            # Avoid duplicate citations
            if citation not in seen:
                print(f"[{i}] {citation}")
                seen.add(citation)
    return 

print_rag_query_answer(
    "What is the impact of LLMs on copyright?",
    vector_dbs["fixed_size"],
    method_name="Fixed Size",
    k=2
)

[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Method: Fixed Size
Question: What is the impact of LLMs on copyright?

Answer:

The impact of LLMs on copyright is significant, as they can have a significant impact on the nature of the copyrighted work. Training data can be informational, expressive, or unpublished within the meaning of copyright law. A very small fraction of training data may be unpublished, and this can have significant implications for copyright law. The ramifications of copyright liability under a pre-training paradigm of AI development are explored in the paper "Break It 'Til You Make It: An Exploration of the Ramifications of Copyright Liability Under a Pre-training Paradigm of AI Development" by Rui-Jie Yew. The paper provides incentives for ongoing technological innovations that prevent infringement, and it emphasizes both technological and scientific progress alongside authorial incentives to create. Training duties for ML systems involve understanding what datasets contain and how they are labeled, and the 

In [47]:
import os

def compare_chunking(query, output_file, k=2):
    output = []

    for method_name, db in vector_dbs.items():
        output.append("\n" + "=" * 80)
        output.append(f"METHOD: {method_name}")
        output.append("=" * 80)

        result = rag_query(
            query,
            db,
            method_name=method_name,
            k=k
        )

        if "error" in result:
            output.append(f"RAG failed: {result['error']}")
            continue

        output.append(f"Question: {result['question']}")
        output.append("\nAnswer:\n")
        output.append(result["answer"])

        output.append("\nREFERENCES:\n")

        seen = set()

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            if citation not in seen:
                output.append(f"[{i}] {citation}")
                seen.add(citation)

        output.append("\nRETRIEVED CHUNKS:")

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")
            page = doc.metadata.get("page", "Unknown")
            chunking = doc.metadata.get("chunking_method", method_name)
            preview = doc.page_content[:300].replace("\n", " ")

            output.append(f"\nChunk {i}")
            output.append(f"Source: {citation}")
            output.append(f"Page: {page}")
            output.append(f"Chunking: {chunking}")
            output.append(f"Preview: {preview}")

    # Ensure directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(output))

    print(f"Results saved to {output_file}")

In [ ]:
q = "What is the impact of LLMs on copyright?"
ofile = "outputs/rag_comparison_test_1.txt"
compare_chunking(q, ofile, k=2)

In [48]:
q = "What do some existing papers say on copyright?"
ofile = "outputs/copyright.txt"
compare_chunking(q, ofile, k=5)

[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en

Results saved to outputs/copyright.txt


In [49]:
q = "What are some policy implications of recent tech?"
ofile = "outputs/policy_implication.txt"
compare_chunking(q, ofile, k=5)

[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Results saved to outputs/policy_implication.txt
